# 🛠️ ISOM 260: The CRAFT Workshop — Ship Three Mini-Apps

**Session 3, Meeting 2 — The Full Craft** | Suffolk University | Prof. Hasan Arslan

---

Last Wednesday your code said hello to a frontier model. Today you ship **actual tools**: an email analyzer, a meeting summarizer, and a product-description generator — each one a real business app in ~15 lines.

The difference between a toy and a tool is the **prompt**. Today's framework: **CRAFT**.

| Letter | Means | The question it answers |
|---|---|---|
| **C** — Context | background & situation | *What does the model need to know?* |
| **R** — Role | who the AI acts as | *Whose expertise should it borrow?* |
| **A** — Action | the specific task | *What exactly should it do?* |
| **F** — Format | output structure | *What shape should the answer take?* |
| **T** — Tone | voice & style | *How should it sound?* |

Run top to bottom. Your `GOOGLE_API_KEY` from Wednesday loads automatically from Colab Secrets.


In [ ]:
# ── Setup (same as Wednesday) ─────────────────────────────────────
!pip install -q google-genai

In [ ]:
# ── Key + client ──────────────────────────────────────────────────
from google.colab import userdata
from google import genai
from google.genai import types

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
MODEL = "gemini-2.5-flash"
print("✅ Workshop open. Let's build.")

## 📐 Part 1 — CRAFT, the function

One function, five ingredients. This exact structure is on your cheat sheet — now it's executable:


In [ ]:
# ── The CRAFT prompt builder ──────────────────────────────────────
def craft_prompt(context: str, role: str, action: str, format: str, tone: str) -> str:
    """Build a well-structured prompt using the CRAFT framework."""
    return f"""[CONTEXT]
{context}

[ROLE]
{role}

[ACTION]
{action}

[FORMAT]
{format}

[TONE]
{tone}"""

def ask(prompt, temperature=0.4):
    """One call, our workshop defaults."""
    r = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=types.GenerateContentConfig(temperature=temperature))
    return r.text

print("✅ craft_prompt() and ask() ready.")

In [ ]:
# ── The before/after that sells the whole framework ──────────────
VAGUE = "write something about our late shipment"

CRAFTED = craft_prompt(
    context="We run an online kitchenware store. A loyal customer's order #4471 (a $340 knife set, anniversary gift) shipped 6 days late due to a warehouse error. It arrives Thursday.",
    role="A senior customer-experience manager who values honesty over corporate speak",
    action="Write an apology email that owns the mistake, gives the Thursday date, and offers a 15% discount code SORRY15",
    format="Subject line, then a 3-short-paragraph email, under 130 words",
    tone="Warm, direct, zero corporate clichés ('we apologize for any inconvenience' is banned)",
)

print("😴 VAGUE PROMPT:\n" + ask(VAGUE)[:400] + "...\n")
print("=" * 60)
print("\n📐 CRAFT PROMPT:\n" + ask(CRAFTED))

**Same model. Same price. Wildly different usefulness.** The model didn't get smarter — *you got specific.* That's the entire discipline of prompt engineering, and you'll now apply it three times.

## 🏗️ Part 2 — The Mini-App Gallery

**Pick ONE to start** (A, B, or C), make it work with the sample input, then — the important part — **swap in your own real input** and tune the CRAFT ingredients until the output is genuinely useful to you. Finish early? Do another, or jump to D.


### 📧 Mini-App A: Email Tone Analyzer
*Before you hit send on that heated reply…*


In [ ]:
# ── A: Email Tone Analyzer ────────────────────────────────────────
def analyze_email(email_text):
    prompt = craft_prompt(
        context=f"A business professional is about to send this email:\n---\n{email_text}\n---",
        role="An executive communication coach who is direct but constructive",
        action="Analyze the tone, flag anything that could land badly, rate send-readiness 1-10, and rewrite it if the rating is below 8",
        format="TONE: one line · RISKS: bullets · RATING: n/10 · REWRITE: (only if needed)",
        tone="Frank, practical, brief",
    )
    return ask(prompt, temperature=0.3)

sample_email = """Hey, I've asked twice now about the Q3 numbers and still nothing.
I need them by Friday or I'm escalating this to Marcus. Not sure what the holdup
is but this is becoming a pattern. - J"""

print(analyze_email(sample_email))
# 🎮 YOUR TURN: paste an email YOU almost sent. Then tune the ROLE — make the
# coach 'blunt New Yorker' vs 'diplomatic HR partner' and compare.

### 📝 Mini-App B: Meeting Notes → Action Items
*The 40-minute meeting, weaponized into 10 lines.*


In [ ]:
# ── B: Meeting Summarizer ─────────────────────────────────────────
def summarize_meeting(raw_notes):
    prompt = craft_prompt(
        context=f"Raw, messy notes from a team meeting:\n---\n{raw_notes}\n---",
        role="A chief of staff famous for ruthless clarity",
        action="Extract decisions made, action items with owners and deadlines, and open questions. Do NOT invent owners or dates that aren't in the notes — mark them ⚠️ UNASSIGNED instead",
        format="DECISIONS: bullets · ACTION ITEMS: [owner] task — deadline · OPEN QUESTIONS: bullets",
        tone="Telegraphic. No filler.",
    )
    return ask(prompt, temperature=0.2)

sample_notes = """ok so launch is moving, sarah thinks oct 15 is too soon, mike agrees,
pushed to nov 1?? karim to check w/ vendor about the pricing page bug (this week)
- budget: we're 12k over, dana said she'd look at cutting the paid ads maybe
- ALSO someone needs to own the demo video before the sales kickoff, tbd
- decided: we're dropping the android beta til Q1"""

print(summarize_meeting(sample_notes))
# 🎮 YOUR TURN: paste real notes (class notes work too). Notice the guardrail in
# ACTION: "do NOT invent owners" — remove it and re-run. See what happens. 👀

### 🛍️ Mini-App C: Product Description Generator
*One product, every channel, thirty seconds.*


In [ ]:
# ── C: Product Description Generator ──────────────────────────────
def product_copy(name, features, audience):
    prompt = craft_prompt(
        context=f"Product: {name}. Features: {', '.join(features)}. Target buyer: {audience}.",
        role="A conversion copywriter who leads with benefits, never feature lists",
        action="Write TWO versions: a website product description (~80 words) and an Instagram caption (~30 words, 2 tasteful emoji max, 3 hashtags)",
        format="WEBSITE: paragraph · INSTAGRAM: caption",
        tone="Confident and concrete — every claim tied to a feature",
    )
    return ask(prompt, temperature=0.8)   # creative task → higher temperature

print(product_copy(
    name="ThermoBrew Travel Mug",
    features=["keeps drinks hot 8 hours", "fits all cupholders", "lifetime warranty", "recycled steel"],
    audience="commuting grad students",
))
# 🎮 YOUR TURN: your side hustle, your club's fundraiser, an imaginary product —
# and notice the temperature is 0.8 here vs 0.2 for meetings. Why?

### 🚀 Mini-App D: Build Your Own
The template is the same every time: *one function, one CRAFT prompt, one real task from your life.* Your week-1 whiteboard is full of candidates.


In [ ]:
# ── D: Yours ──────────────────────────────────────────────────────
def my_app(user_input):
    prompt = craft_prompt(
        context=f"...your situation...\nINPUT:\n{user_input}",
        role="...whose expertise?...",
        action="...do exactly what?...",
        format="...what shape?...",
        tone="...sounding how?...",
    )
    return ask(prompt)

# print(my_app("your input here"))

## 💬 Part 3 — Multi-Turn: Give It a Memory

Every call so far was **amnesiac** — the model forgets you between calls. Real assistants hold a conversation. The SDK does the bookkeeping:


In [ ]:
# ── A conversation that remembers ─────────────────────────────────
chat = client.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(
        system_instruction="You are a pragmatic business mentor for a college student. Two-sentence answers."),
)

print("Q1:", chat.send_message("I'm thinking of starting a meal-prep service for students.").text)
print("\nQ2:", chat.send_message("What's the biggest risk with that idea?").text)
print("\nQ3:", chat.send_message("Ok — give me one concrete first step for it.").text)

# "that idea"... "it"... — the model resolved those because the chat RESENDS the
# whole history every turn. Remember Wednesday: the context window is the memory,
# and you're billed for it each time. That's why long chats get slow and pricey.

## 💰 Part 4 — What would your app cost for real?

You built a tool today. Would it survive a CFO?


In [ ]:
# ── Price your mini-app at business scale ─────────────────────────
# We measure one REAL call (the email analyzer) and scale it up.
resp = client.models.generate_content(model=MODEL, contents=craft_prompt(
    context=f"Email:\n{sample_email}", role="communication coach",
    action="analyze tone and rewrite", format="short sections", tone="frank"))
u = resp.usage_metadata
IN_PRICE, OUT_PRICE = 0.30, 2.50    # $ per 1M tokens, Flash-class paid tier
per_call = u.prompt_token_count/1e6*IN_PRICE + u.candidates_token_count/1e6*OUT_PRICE

CALLS_PER_DAY = 500                  # <- your imagined business volume
print(f"tokens per call: {u.prompt_token_count} in / {u.candidates_token_count} out")
print(f"cost per call:   ${per_call:.5f}")
print(f"at {CALLS_PER_DAY}/day: ${per_call*CALLS_PER_DAY:.2f}/day → ${per_call*CALLS_PER_DAY*30:.2f}/month")
print(f"\nAn employee-hour costs more than this app's month. That's the business case.")

## ✅ What you shipped today

Three working business tools, a conversation with memory, and a CFO-ready cost estimate — all from **one function and five ingredients**. CRAFT is now muscle memory; the cheat sheet on the course site is your permanent reference.

**Reminders:**
- 🧭 **HW #4 (Tokenizer Safari)** — due **Thursday Oct 1, 11:59 PM** on Canvas
- 🕵️ Haven't tried the **Job Hunter** sneak peek yet? Your key works on it now.

**Wednesday — Session 4:** the training wheels come off. Your code stops *telling* the model what to do, and the model starts **deciding for itself** — which tool to use, when, and whether it's done. Your first real agent. 🤖
